This NB is to learn how to implement multi-threading for a toy example <br/>
Examples are taken from here: https://docs.julialang.org/en/v1/manual/multi-threading/#man-multithreading

In [10]:
using BenchmarkTools

In [4]:
function sum_single(nums)
    sum = 0
    for i in nums 
       sum += i
    end
    return sum
end

sum_single (generic function with 1 method)

In [102]:
function sum_single(a)
   s = 0
   for i in a
       s += i
   end
   s
end

@benchmark sum_single(1:100_000_000)

BenchmarkTools.Trial: 10000 samples with 1000 evaluations per sample.
 Range (min … max):  2.600 ns … 178.700 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     6.100 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   6.951 ns ±   4.705 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▄▂▁▁▁▂     ▃█▄▁ ▃▂                                        ▄ ▁
  ██████▇▃▄▄▅████▁██▇▇▆▁▃▄▁▁▃▁▁▁▁▃▁▃▁▁▁▁▃▁▅▅▆▅▃▄▄▃▃▁▁▃▃▁▄▁▁▁█ █
  2.6 ns       Histogram: log(frequency) by time      19.5 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [98]:
function sum_multi_good(a)
    chunks = Iterators.partition(a, length(a) ÷ Threads.nthreads())
    tasks = Vector{Task}(undef, length(chunks))
    i = 1
    for chunk in chunks
        tasks[i] = Threads.@spawn sum_single(chunk)
        i += 1
    end
    chunk_sums = fetch.(tasks)
    return sum_single(chunk_sums)
end

sum_multi_good (generic function with 1 method)

In [103]:
@benchmark sum_multi_good(1:100_000_000)

BenchmarkTools.Trial: 10000 samples with 7 evaluations per sample.
 Range (min … max):  2.586 μs … 867.814 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     7.957 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   9.469 μs ±  19.265 μs  ┊ GC (mean ± σ):  2.54% ± 1.84%

      ▁▃▃▃▆█▇▆▅▅▅▄▃▂▁▁▁▁▁                                     ▂
  ▄▆▇▆██████████████████████▇▇▇▅▅▅▆▄▄▅▅▄▁▄▁▄▅▄▅▆▃▄▄▃▃▅▆▄▃▄▄▁▅ █
  2.59 μs      Histogram: log(frequency) by time      33.1 μs <

 Memory estimate: 2.89 KiB, allocs estimate: 34.

In [91]:
a = 1:10
chunks = Iterators.partition(a, length(a) ÷ Threads.nthreads())
tasks = Vector{Task}(undef, length(chunks))
i = 1
for chunk in chunks
    tasks[i] = Threads.@spawn sum_single(chunk)
    i += 1
end
chunk_sums = fetch.(tasks)
#return sum_single(chunk_sums)

5-element Vector{Int64}:
  3
  7
 11
 15
 19

In [94]:
tasks[1]

Task (done) @0x0000017c50d3a100

In [90]:
collect(chunks)

5-element Vector{UnitRange{Int64}}:
 1:2
 3:4
 5:6
 7:8
 9:10

In [11]:
@btime sum_single(1:1_000_000)

  2.500 ns (0 allocations: 0 bytes)


500000500000

In [12]:
typeof(1:5)

UnitRange{Int64}

In [19]:
typeof(1:1:10)

StepRange{Int64, Int64}

In [21]:
my_range = 1:10

1:10

In [22]:
my_range.start

1

In [20]:
fieldnames(UnitRange)

(:start, :stop)

In [14]:
sin.(1:5)

5-element Vector{Float64}:
  0.8414709848078965
  0.9092974268256817
  0.1411200080598672
 -0.7568024953079282
 -0.9589242746631385

In [16]:
typeof(0.5:5)

StepRangeLen{Float64, Base.TwicePrecision{Float64}, Base.TwicePrecision{Float64}, Int64}

In [24]:
data = 1:1000
result = collect(sqrt.(data .+ 1));

In [26]:
data = 1:1000                     # Lazy range, no full array allocated
step1 = data .+ 1                # Broadcast creates a new array here, so not lazy by default

# But if you use Iterators.map:
step1_lazy = Iterators.map(x -> x + 1, data)  # No array created here

step2_lazy = Iterators.map(sqrt, step1_lazy)  # Still lazy; only computes when iterated

result = collect(step2_lazy);     # Only here all computations happen and array is created

In [33]:
range2 = data .^2;

In [31]:
typeof(range2)

StepRangeLen{Float64, Float64, Float64, Int64}

In [38]:
f(x) = 2*x + 0.5
g(x) = 2*x 

g (generic function with 1 method)

In [37]:
2 .* data .+ 0.5

2.5:2.0:2000.5

In [42]:
typeof(1:0.1:10)

StepRangeLen{Float64, Base.TwicePrecision{Float64}, Base.TwicePrecision{Float64}, Int64}

In [58]:
nums = 1.08:0.25:2.5

1.08:0.25:2.33

In [54]:
fieldnames(typeof(nums))

(:ref, :step, :len, :offset)

In [77]:
r = Base.StepRangeLen(10, 0.1, 11)

10.0:0.1:11.0

In [85]:
x = 2:2:10

x .^2 

5-element Vector{Int64}:
   4
  16
  36
  64
 100

In [79]:
r.offset

1

In [70]:
fieldnames(typeof(r))

(:ref, :step, :len, :offset)

In [69]:
typeof(r)

StepRangeLen{Float64, Float64, Float64, Int64}

In [105]:
using Base.Threads
using BenchmarkTools

# Correctly flatten the pairs into a 1D Vector of tuples
function get_pairs(ivals, jvals)
    pairs = Vector{Tuple{eltype(ivals), eltype(jvals)}}()
    for i in ivals
        for j in jvals
            push!(pairs, (i, j))
        end
    end
    return pairs
end

# Non-parallel version
function serial_foo(ivals, jvals, pars)
    pairs = get_pairs(ivals, jvals)
    for idx in 1:length(pairs)
        i, j = pairs[idx]
        foo(i, j, pars)
    end
end

# Parallel version
function parallel_foo(ivals, jvals, pars)
    pairs = get_pairs(ivals, jvals)
    @threads for idx in 1:length(pairs)
        i, j = pairs[idx]
        foo(i, j, pars)
    end
end

# Example foo for demo
function foo(i, j, pars)
    s = 0.0
    for k in 1:1000
        s += sqrt(k + i + j)
    end
    return s
end

# Sample parameters
ivals = 1:10
jvals = 1:10
pars = nothing

println("Benchmarking serial version:")
@btime serial_foo($ivals, $jvals, $pars)

println("Benchmarking parallel version:")
@btime parallel_foo($ivals, $jvals, $pars)


Benchmarking serial version:
  39.200 μs (4 allocations: 3.67 KiB)
Benchmarking parallel version:
  13.700 μs (25 allocations: 6.19 KiB)
